# 🔄 Agent-to-Agent: Hierarchical Research with Bigdata.com

This notebook shows a **hierarchical agent** that checks internal data first, then escalates to the **Bigdata.com Research Agent** for deep, cited research when needed.

## What This Demonstrates

**Internal-first flow:**
- **Primary Agent** queries internal DB (portfolios, holdings, transactions) and internal research (FAISS vector store).
- **Escalation** to Bigdata.com Research Agent only when the question needs external, multi-source analysis (20–60s, full citations).

**Research Agent:**
- Multi-step reasoning with RAG; answers include inline citations [1], [2] and numbered sources.
- Retry, stream timeout, and logging (production-ready; see `research_client.py`).

**Framework flexibility:**
> This demo uses **LangChain** and **LangSmith**. The pattern—internal tools first, then one “research” tool—works with **CrewAI**, **AutoGen**, or custom graphs. The key is tool ordering and system prompt that prefers internal sources.

## Architecture

![Agent to Bigdata Research Agent](./static/agent_to_research_small.jpg)

**Key benefits:** Cost and latency control (internal answers are fast); full citations when escalating; reusable `langgraph_core` setup and tools.

## Use Cases

| Role | Example Questions |
|------|-------------------|
| **Equity Research** | Thesis validation, competitive analysis |
| **Credit Research** | Covenant analysis, refinancing risks |
| **Credit Risk** | Counterparty exposure, default drivers |

---

## Langsmith Tracing

![LangSmith Tracing](./static/langsmith.png)

## 1️⃣ Install Dependencies

> **Note:** If you're using `uv` to manage dependencies (recommended), you can skip the pip install cell below. Dependencies are already installed via `uv pip install -r requirements.txt`.

In [1]:
# Dependencies are installed via uv: uv pip install -r requirements.txt
# %pip install langchain langchain-openai langchain-community faiss-cpu requests python-dotenv -q

## 2️⃣ Setup Environment

In [2]:
from langgraph_core import setup_environment

# Setup with LangSmith tracing
config = setup_environment(
    langsmith_project='bigdata-agent-to-agent',
    enable_tracing=True
)

print("\n🔗 View agent traces: https://smith.langchain.com")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ LangSmith tracing enabled → Project: bigdata-agent-to-agent
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...

🔗 View agent traces: https://smith.langchain.com


## 3️⃣ Initialize Data Sources

Create the internal database and vector store with sample financial data.

In [3]:
from langgraph_core import create_financial_database, create_vector_store

# Create SQLite database with portfolios, holdings, transactions
create_financial_database()

# Create vector store with internal research documents
create_vector_store()

print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")

✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)


## 4️⃣ Create Hierarchical Agent

This agent is configured to:
1. Check **internal sources first** (faster, proprietary data)
2. Use **quick external lookups** for company info
3. **Escalate** to Research Agent only when needed

In [4]:
from langgraph_core import create_hierarchical_agent

# Create agent with hierarchical tool priority
agent = create_hierarchical_agent(include_research_agent=True)

✅ Hierarchical agent created with 5 tools:
   Internal: ['internal_query_database', 'internal_portfolio_summary', 'internal_search_research']
   External: ['bigdata_lookup_company', 'bigdata_research_agent']


---

# 📈 Equity Research Use Cases

Questions that equity analysts typically ask.

### Query 1: Internal Holdings Check (No Escalation Expected)

Simple portfolio question - should use only internal tools.

In [5]:
from langgraph_core import display_agent_response

display_agent_response(agent, """
What is our total exposure to NVIDIA across all portfolios? 
Include position sizes, average cost basis, and unrealized P&L.
""", show_json=True)

🔧 internal_query_database: {
  "sql_query": "SELECT portfolio_id, shares, avg_cost, market_value, unrealized_pnl FROM holdings WHERE company_name = 'NVIDIA'"
}...


We currently have no exposure to NVIDIA across any of our portfolios. There are no holdings of NVIDIA in our internal database.

{'response': 'We currently have no exposure to NVIDIA across any of our portfolios. There are no holdings of NVIDIA in our internal database.',
 'tools_count': 1,
 'tools': [{'name': 'internal_query_database',
   'args': {'sql_query': "SELECT portfolio_id, shares, avg_cost, market_value, unrealized_pnl FROM holdings WHERE company_name = 'NVIDIA'"}}],
 'tool_results': ['{\n  "query": "SELECT portfolio_id, shares, avg_cost, market_value, unrealized_pnl FROM holdings WHERE company_name = \'NVIDIA\'",\n  "row_count": 0,\n  "results": []\n}']}

### Query 2: Investment Thesis Validation (Internal + External)

Combines internal research with market data - may escalate for current news.

In [6]:
display_agent_response(agent, """
Review our investment thesis for NVIDIA:
1. What does our internal research say about NVIDIA's competitive position?
2. What recent market developments might affect this thesis?
3. Should we adjust our position based on current information?
""")

🔧 internal_search_research: {
  "query": "NVIDIA competitive position",
  "top_k": 3
}...
🔧 internal_query_database: {
  "sql_query": "SELECT * FROM holdings WHERE company_name = 'NVIDIA'"
}...
🔧 bigdata_research_agent: {
  "query": "Recent market developments affecting NVIDIA",
  "research_effort": "standard"
}...


### NVIDIA's Competitive Position

According to our internal research, NVIDIA maintains a strong competitive position in the semiconductor space, particularly driven by its dominance in AI and data center markets. Key points from our investment thesis include:

1. **Data Center Revenue**: NVIDIA's data center revenue has seen a significant increase, driven by high demand for its H100/H200 GPUs used in AI training.
2. **Next-Gen Architecture**: The upcoming Blackwell Architecture (B100/B200 GPUs) is expected to offer 2.5x performance improvements, launching in Q2 2025.
3. **Software Ecosystem**: NVIDIA's CUDA ecosystem, with over 4 million developers, creates substantial switching costs, reinforcing its competitive moat.
4. **AI Inference Market**: The AI inference market is projected to reach a $150 billion total addressable market by 2027, as enterprises scale AI deployments.
5. **Risk Factors**: Potential risks include export restrictions to China, competition from AMD, and supply constraints.

The internal strategy memo also suggests increasing NVIDIA's portfolio weight by 3% due to strong AI training demand [From internal research].

### Recent Market Developments

Recent market developments affecting NVIDIA include:

1. **Financial Performance**: NVIDIA reported record revenue of $57.0 billion for Q3 fiscal 2026, marking a 22% increase from the previous quarter [2][3].
2. **Strategic Partnerships**: NVIDIA has invested $2 billion in CoreWeave to expand AI infrastructure and partnered with Mercedes-Benz for robotaxi services [7][10].
3. **Supply Chain**: NVIDIA has allocated 70% of its HBM4 demand to SK Hynix, ensuring a stable supply for its AI accelerators [11].
4. **Stock Performance**: NVIDIA's stock has shown substantial growth, with a 57.23% increase over the last year, and analysts expect continued strong performance [13][14].

### Position Adjustment Recommendation

Given NVIDIA's strong competitive position, recent strategic partnerships, and robust financial performance, it may be prudent to maintain or even increase our position in NVIDIA. The internal strategy memo already recommends a 3% increase in portfolio weight for NVIDIA, aligning with the positive market developments and growth prospects in AI and data centers. However, it's essential to monitor potential risks such as geopolitical tensions and supply chain constraints that could impact future performance.

### Query 3: Competitive Analysis (Escalation Expected)

Deep competitive analysis requires external research agent.

In [7]:
display_agent_response(agent, """
Perform a competitive analysis of AMD vs NVIDIA in the AI accelerator market:

1. First check: Do we have any internal research comparing these companies?
2. Then analyze: Current Market share trends, product roadmaps, and customer wins
3. Conclude: Which company is better positioned for 2025-2026?

Cite specific sources for your analysis.
""")

🔧 internal_search_research: {
  "query": "AMD vs NVIDIA AI accelerator market",
  "top_k": 3
}...
🔧 bigdata_research_agent: {
  "query": "Current market share trends, product roadmaps, and customer wins for AMD and NVIDIA in the AI accelerator market",
  "research_effort": ...


### Internal Research Insights

From our internal research, we have the following insights:

1. **AMD**:
   - The MI300X GPU offers 192GB HBM3 memory, which is 1.5 times that of NVIDIA's H100, and shows strong inference performance for large language model (LLM) workloads. It has confirmed deployments with Microsoft Azure and Oracle Cloud, targeting over $5 billion in AI GPU revenue by 2025.
   - AMD's EPYC server CPUs have gained significant market share, reaching over 33% from just 5% in 2018, with the Turin (Zen 5) launching in H1 2025 with 192 cores.
   - Challenges include a lagging ROCm software ecosystem compared to NVIDIA's CUDA and NVIDIA's strong mindshare among AI developers.

2. **NVIDIA**:
   - NVIDIA's data center revenue reached $18.4 billion, a 409% year-over-year increase, driven by demand for H100/H200 GPUs for AI training.
   - The upcoming Blackwell architecture (B100/B200 GPUs) is expected to launch in Q2 2025, offering 2.5 times the performance.
   - NVIDIA's CUDA ecosystem, with over 4 million developers, creates significant switching costs.
   - Risks include China export restrictions, AMD competition, and supply constraints.

### External Research Analysis

#### Market Share Trends
- **NVIDIA**: Dominates the AI accelerator market with a market share ranging from 70% to 95% in 2023-2024, expected to maintain 70-75% through 2030. This is supported by their H100 and upcoming H200 and Blackwell series [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11].
- **AMD**: Gaining market share, with a 179% growth in its data center GPU segment between 2023 and 2024, increasing its market share from 3% to 4%. Analysts predict AMD could capture around 10% of the AI chip market by 2025-2026 [15, 16, 17, 18, 19, 20].

#### Product Roadmaps
- **AMD**: Aggressive roadmap with the MI300 series currently in use, and future MI350, MI400, and MI500 series planned through 2027, offering significant performance improvements [33, 34, 35, 36, 37, 38, 39, 40].
- **NVIDIA**: Continues its aggressive update cycle with the Hopper, Blackwell, and upcoming Rubin platforms, focusing on generative AI and accelerated computing [41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62].

#### Customer Wins
- **AMD**: Secured major wins with OpenAI, Microsoft, Meta Platforms, Oracle, and IBM, among others, for its Instinct MI300 and MI400 series [63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75].
- **NVIDIA**: Extensive customer base including Oracle, Google Cloud, AWS, Microsoft, and various AI model developers and hardware partners [76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88].

### Conclusion: Positioning for 2025-2026

**NVIDIA** is currently better positioned due to its dominant market share, robust product roadmap, and extensive ecosystem with significant developer support. However, **AMD** is rapidly gaining ground with competitive products and strategic customer wins, potentially capturing a significant portion of the market by 2025-2026. The choice between the two may depend on specific use cases, cost considerations, and ecosystem compatibility.

---

# 💳 Credit Research Use Cases

Questions relevant to fixed income and credit analysts.

### Query 4: Debt and Leverage Analysis (Escalation Expected)

Credit-focused questions typically need external data.

In [8]:
display_agent_response(agent, """
Analyze the credit profile of Intel Corporation:

1. Current debt levels and maturity schedule
2. Cash flow coverage ratios and liquidity position
3. Recent credit rating actions or outlook changes
4. Key risks that could affect their investment-grade status

Note: We don't hold Intel, so you may need external sources.
""")

🔧 bigdata_research_agent: {
  "query": "Analyze the credit profile of Intel Corporation, including current debt levels and maturity schedule, cash flow coverage ratios and liqu...


Intel Corporation's credit profile is currently under pressure due to increased leverage and strategic investments, which have led to recent downgrades by major credit rating agencies.

### Current Debt Levels and Maturity Schedule
- **Total Debt:** As of December 31, 2022, Intel's total debt was $42.051 billion, with $4.367 billion in short-term debt and $37.684 billion in long-term debt [1].
- **Debt Maturities:** The debt maturities are scheduled as follows [1]:
  - 2023: $423 million
  - 2024: $2,288 million
  - 2025: $3,750 million
  - 2026: $1,000 million
  - 2027: $3,826 million
  - 2028 and thereafter: $27,998 million
- **Recent Debt Levels:** By Q3 2024, total debt increased to $50.236 billion [2].

### Cash Flow Coverage Ratios and Liquidity Position
- **Operating Cash Flow:** For fiscal year 2022, Intel's net cash from operating activities was $15.433 billion [1]. For the nine months ended September 30, 2023, it was $6.847 billion [6].
- **Liquidity Position:** As of December 31, 2022, Intel had $11.144 billion in cash and $17.194 billion in short-term investments, totaling $28.801 billion [1]. By September 28, 2024, this was $24.086 billion [2].
- **Credit Facilities:** Intel has a commercial paper program and revolving credit facilities totaling $15 billion, with no outstanding borrowings as of the latest reports [1][5].

### Recent Credit Rating Actions or Outlook Changes
- **S&P Global Ratings:** Downgraded Intel from 'BBB+' to 'BBB' in December 2024, citing slow business recovery [7][8].
- **Moody's:** Downgraded Intel to 'Baa1' in August 2024 and further to 'Baa2' in August 2025 due to weak profitability [10][13].
- **Fitch Ratings:** Downgraded Intel from 'BBB+' to 'BBB' with a negative outlook in August 2025, highlighting execution risks and elevated debt levels [15][16][20].

### Key Risks Affecting Investment-Grade Status
- **Market Demand and Competition:** Intel faces intense competition from AMD, Qualcomm, and others, impacting demand and pricing [1].
- **Execution Risks:** Challenges in executing the IDM 2.0 strategy and manufacturing expansions pose significant risks [1].
- **Elevated Debt Levels:** High capital expenditures and debt levels increase financial leverage and risk [1][20].
- **Supply Chain Vulnerabilities:** Dependence on a complex supply chain exposes Intel to disruptions [1].
- **Geopolitical and Regulatory Risks:** Trade policies and geopolitical tensions could impact costs and market access [1].
- **Product Defects and Cybersecurity:** Risks related to product defects and security vulnerabilities could lead to significant costs and reputational damage [1].

In summary, Intel's credit profile is challenged by increased debt, strategic execution risks, and competitive pressures, as reflected in recent credit rating downgrades.

### Query 5: Sector-Wide Credit Trends (Escalation Expected)

Macro credit analysis requires deep research.

In [9]:
display_agent_response(agent, """
What are the key credit risks facing the semiconductor sector in the current environment?

Consider:
- Capital expenditure requirements and debt financing
- Cyclical demand patterns and inventory corrections
- Geopolitical risks (US-China tensions, export controls)
- Recent bond issuances or refinancing activity

Which semiconductor companies are most vulnerable from a credit perspective?
""")

🔧 bigdata_research_agent: {
  "query": "Analyze the key credit risks facing the semiconductor sector, focusing on capital expenditure requirements, debt financing, cyclical dem...


The semiconductor sector faces several key credit risks in the current environment:

1. **Capital Expenditure (CapEx) Requirements**: The industry is highly capital-intensive, requiring substantial investments in R&D and manufacturing facilities. Companies like TSMC and Micron are planning significant increases in CapEx, which can strain cash flows and increase reliance on external financing [1][2][3]. Intel, for example, faces pressure from these demands as it undergoes large-scale transformations [5].

2. **Debt Financing**: Many semiconductor companies rely on significant debt to fund their CapEx and R&D needs. High debt obligations can reduce financial flexibility and increase the risk of credit rating downgrades. Intel has substantial debt obligations, while Microchip Technology has a debt-to-equity ratio surpassing industry norms, indicating potential financial challenges [6][7][8][11].

3. **Cyclical Demand Patterns**: The industry is cyclical, experiencing periods of rapid growth followed by downturns. This volatility makes it challenging for companies to maintain consistent profitability, with even major players experiencing swings from profits to losses [6][12][13][14][15][16][18].

4. **Inventory Corrections**: Excess stock in the supply chain can lead to reduced orders and production. Companies like ON Semiconductor have been affected by such corrections, impacting revenues and potentially leading to inventory write-downs [19][20][21].

5. **Geopolitical Risks**: US-China tensions and export controls significantly disrupt the semiconductor supply chain. The US has imposed export controls to restrict China's access to advanced semiconductors, while China has retaliated with restrictions on critical raw materials. These actions lead to supply chain fragmentation and increased costs [23][24][25][26][27][28][29][30][31][32][33][34][35].

6. **Recent Bond Issuances or Refinancing Activity**: The sector has seen various bond issuances and refinancing activities, indicating active debt management. Companies like SMIC and ON Semiconductor have engaged in such activities to manage their financial needs [36][37][38][39][40][41][42][43][44][45].

**Most Vulnerable Semiconductor Companies from a Credit Perspective**:

- **Intel Corp.**: Faces low headroom under current ratings due to its transformation efforts and significant CapEx demands, coupled with substantial debt obligations [5].
- **Microchip Technology**: Has a high debt-to-equity ratio, suggesting a substantial debt burden and potential financial challenges [11].
- **ON Semiconductor**: Affected by inventory corrections and has a notable debt-to-equity ratio, indicating potential financial strain [20].

Companies with high debt loads, significant CapEx demands, poor inventory management, and substantial geopolitical exposure are particularly vulnerable from a credit perspective.

### Query 6: Refinancing Risk Assessment

In [10]:
display_agent_response(agent, """
Which companies in our portfolios might face refinancing challenges in 2025-2026?

1. First, check what companies we hold across all portfolios
2. For each major holding, assess:
   - Debt maturity walls
   - Current interest coverage
   - Access to capital markets
3. Flag any companies with elevated refinancing risk
""")

🔧 internal_query_database: {
  "sql_query": "SELECT DISTINCT company_name FROM holdings"
}...
🔧 bigdata_research_agent: {
  "query": "Assess the refinancing risk for Apple Inc. in 2025-2026, focusing on debt maturity walls, current interest coverage, and access to capit...
🔧 bigdata_research_agent: {
  "query": "Assess the refinancing risk for Microsoft Corporation in 2025-2026, focusing on debt maturity walls, current interest coverage, and acce...
🔧 bigdata_research_agent: {
  "query": "Assess the refinancing risk for Alphabet Inc. in 2025-2026, focusing on debt maturity walls, current interest coverage, and access to ca...
🔧 bigdata_research_agent: {
  "query": "Assess the refinancing risk for Amazon.com Inc. in 2025-2026, focusing on debt maturity walls, current interest coverage, and access to ...
🔧 bigdata_research_agent: {
  "query": "Assess the refinancing risk for Meta Platforms Inc. in 2025-2026, focusing on debt maturity walls, current interest coverage, and access...
🔧 bigd

Here's an assessment of the refinancing risks for companies in our portfolios for 2025-2026:

1. **Apple Inc.**: Low refinancing risk due to strong liquidity, excellent credit ratings, and proactive debt management. [1]

2. **Microsoft Corporation**: Very low refinancing risk with strong interest coverage and top-tier credit ratings, ensuring excellent access to capital markets. [2]

3. **Alphabet Inc.**: Very low refinancing risk with an impressive interest coverage ratio and excellent access to capital markets. [3]

4. **Amazon.com Inc.**: Low refinancing risk with manageable debt maturities, strong interest coverage, and excellent access to capital markets. [4]

5. **Meta Platforms Inc.**: Very low refinancing risk with no significant debt maturities in the near term, strong interest coverage, and excellent access to capital markets. [5]

6. **NVIDIA Corporation**: Very low refinancing risk with strong profitability, low leverage, and high liquidity. [6]

7. **Advanced Micro Devices**: Low refinancing risk with strong interest coverage, healthy liquidity, and low debt burden. [7]

8. **Broadcom Inc.**: Manageable refinancing risk with strong interest coverage and demonstrated access to capital markets. [8]

9. **Taiwan Semiconductor**: Very low refinancing risk with exceptional interest coverage and significant cash reserves. [9]

10. **Palantir Technologies**: Very low refinancing risk due to minimal debt and strong liquidity. [10]

11. **Salesforce Inc.**: Strong interest coverage and liquidity suggest good access to capital markets, but further details on debt maturities are needed for a comprehensive assessment. [11]

12. **Oracle Corporation**: Elevated refinancing risk due to substantial debt maturities, a deteriorating credit outlook, and growing investor caution related to its AI investments. [12]

Overall, most companies in our portfolios have low refinancing risks, except for Oracle Corporation, which faces elevated risks due to its aggressive expansion and significant debt maturities.

---

# ⚠️ Credit Risk Analyst Use Cases

Questions focused on counterparty risk and default analysis.

### Query 7: Counterparty Exposure Analysis (Internal First)

In [11]:
display_agent_response(agent, """
Calculate our total counterparty exposure by company:

1. Sum up all positions across portfolios for each ticker
2. Calculate as percentage of total AUM
3. Identify our top 5 concentration risks
4. Flag any single-name exposures exceeding 15% of total

Present in a risk report format.
""")

🔧 internal_query_database: {
  "sql_query": "SELECT portfolio_id, ticker, company_name, shares, current_price, market_value FROM holdings"
}...
🔧 internal_query_database: {
  "sql_query": "SELECT SUM(aum) as total_aum FROM portfolios"
}...


### Risk Report: Counterparty Exposure Analysis

#### Total Counterparty Exposure by Company

1. **Apple Inc. (AAPL)**
   - Total Market Value: $7,410,000
   - Percentage of Total AUM: 7.80%

2. **Microsoft Corporation (MSFT)**
   - Total Market Value: $9,556,500
   - Percentage of Total AUM: 10.06%

3. **NVIDIA Corporation (NVDA)**
   - Total Market Value: $17,546,000
   - Percentage of Total AUM: 18.47%

4. **Alphabet Inc. (GOOGL)**
   - Total Market Value: $876,250
   - Percentage of Total AUM: 0.92%

5. **Amazon.com Inc. (AMZN)**
   - Total Market Value: $1,354,500
   - Percentage of Total AUM: 1.43%

6. **Meta Platforms Inc. (META)**
   - Total Market Value: $2,632,500
   - Percentage of Total AUM: 2.77%

7. **Advanced Micro Devices (AMD)**
   - Total Market Value: $1,162,000
   - Percentage of Total AUM: 1.22%

8. **Broadcom Inc. (AVGO)**
   - Total Market Value: $2,137,500
   - Percentage of Total AUM: 2.25%

9. **Taiwan Semiconductor (TSM)**
   - Total Market Value: $557,250
   - Percentage of Total AUM: 0.59%

10. **Palantir Technologies (PLTR)**
    - Total Market Value: $1,631,250
    - Percentage of Total AUM: 1.72%

11. **Salesforce Inc. (CRM)**
    - Total Market Value: $3,255,000
    - Percentage of Total AUM: 3.43%

12. **Oracle Corporation (ORCL)**
    - Total Market Value: $2,103,000
    - Percentage of Total AUM: 2.21%

#### Top 5 Concentration Risks

1. **NVIDIA Corporation (NVDA)** - 18.47%
2. **Microsoft Corporation (MSFT)** - 10.06%
3. **Apple Inc. (AAPL)** - 7.80%
4. **Salesforce Inc. (CRM)** - 3.43%
5. **Meta Platforms Inc. (META)** - 2.77%

#### Single-Name Exposures Exceeding 15%

- **NVIDIA Corporation (NVDA)**: 18.47% of total AUM

**Note:** The exposure to NVIDIA Corporation exceeds the 15% threshold, indicating a significant concentration risk that should be monitored closely.

### Query 8: ESG/Regulatory Risk (Escalation Expected)

In [12]:
display_agent_response(agent, """
Assess regulatory and ESG risks for our technology holdings:

1. Which of our holdings face significant regulatory scrutiny?
2. What recent regulatory developments could impact valuations?
3. Are there any ESG controversies affecting our portfolio companies?
4. Which positions should we consider reducing due to regulatory risk?

Focus on antitrust, data privacy, and AI governance regulations.
""")

🔧 internal_portfolio_summary: {
  "portfolio_id": "PF003"
}...
🔧 internal_search_research: {
  "query": "regulatory scrutiny and ESG controversies for NVIDIA, Microsoft, Apple, Salesforce, Oracle",
  "top_k": 3
}...
🔧 bigdata_research_agent: {
  "query": "Recent regulatory developments impacting NVIDIA, Microsoft, Apple, Salesforce, Oracle in antitrust, data privacy, and AI governance",
  ...


### Regulatory and ESG Risks for Technology Holdings

#### 1. Significant Regulatory Scrutiny
- **NVIDIA**: Faces antitrust scrutiny, particularly from China, related to its Mellanox acquisition. The US Department of Justice is also investigating potential antitrust violations concerning NVIDIA's AI market dominance [1].
- **Microsoft**: Under EU scrutiny for bundling its Teams app with Office products. Switzerland and Brazil are investigating potential anti-competitive practices [2].
- **Apple**: Faces antitrust lawsuits in the US and scrutiny in Europe, India, and China over its App Store policies [3].
- **Salesforce**: Under DOJ investigation concerning its acquisition of Slack [4].
- **Oracle**: Under scrutiny for its role in tuition pricing algorithms and Java licensing enforcement [5].

#### 2. Recent Regulatory Developments Impacting Valuations
- **NVIDIA**: US export controls on AI chips impact business in China. China approved imports of NVIDIA H200 AI chips under specific conditions [6].
- **Microsoft**: Avoided a large EU fine by unbundling Teams from Office products [7].
- **Apple**: Fined €500 million in the EU for non-compliance with the Digital Markets Act [8].
- **Salesforce**: Faces evolving antitrust laws that could impact operations and acquisitions [9].
- **Oracle**: Engaged in litigation concerning GDPR violations [10].

#### 3. ESG Controversies
- **NVIDIA**: Involved in a class-action lawsuit for allegedly using pirated books to train AI models [11].
- **Microsoft**: Faces complaints about facilitating the deletion of surveillance data, potentially violating GDPR [12].
- **Apple**: Fined for its App Tracking Transparency feature, impacting third-party developers [13].
- **Salesforce**: Experienced a data exposure incident related to Gainsight-published applications [14].
- **Oracle**: Denies allegations of massive data theft from its Cloud [15].

#### 4. Positions to Consider Reducing Due to Regulatory Risk
- **NVIDIA**: Due to significant antitrust scrutiny and export control impacts, consider monitoring closely for potential reduction [16].
- **Apple**: Given ongoing antitrust lawsuits and fines, evaluate the risk-reward balance [17].
- **Salesforce**: With DOJ investigations and evolving antitrust laws, assess the potential impact on growth [18].

These insights highlight the need for continuous monitoring of regulatory developments and ESG controversies to manage risks effectively.

---

# 🔬 Custom Query

Try your own hierarchical research query:

In [13]:
# Enter your own query
custom_query = """
Compare our internal research on NVIDIA with the latest market sentiment.
Is our thesis still valid?
"""

display_agent_response(agent, custom_query)

🔧 internal_search_research: {
  "query": "NVIDIA investment thesis",
  "top_k": 3
}...
🔧 bigdata_research_agent: {
  "query": "latest market sentiment on NVIDIA",
  "research_effort": "standard"
}...


### Internal Research on NVIDIA

Our internal investment thesis on NVIDIA, updated in December 2024, highlights several key points:

1. **Data Center Revenue**: NVIDIA's data center revenue reached $18.4 billion, marking a 409% year-over-year increase, driven by strong demand for H100/H200 GPUs used in AI training.
2. **Next-Gen Architecture**: The upcoming Blackwell Architecture (B100/B200 GPUs) is expected to launch in Q2 2025, offering 2.5x performance improvements.
3. **Software Ecosystem**: NVIDIA's CUDA ecosystem, with over 4 million developers, creates significant switching costs, reinforcing its competitive moat.
4. **AI Inference Market**: The AI inference market is projected to reach a $150 billion total addressable market (TAM) by 2027 as enterprises scale AI deployments.

**Risk Factors**: Potential risks include China export restrictions, competition from AMD, and supply constraints.

**Price Target**: $950 (25x FY26E EPS) with a "STRONG BUY" rating.

### Latest Market Sentiment on NVIDIA

The latest market sentiment on NVIDIA is highly positive, driven by its leadership in AI and strong financial performance:

1. **Analyst Ratings**: NVIDIA is widely recommended as a "Buy" or "Strong Buy" by analysts, with a consensus price target of $265.21, suggesting a 33-40% upside from its current price [2][3].
2. **Financial Performance**: NVIDIA reported a 114.2% year-over-year revenue increase to $130.50 billion in fiscal year 2025, with net income up 144.9% to $72.88 billion [5][6].
3. **Growth Drivers**: Key growth areas include AI dominance, data center expansion, strategic investments, and continuous innovation in AI models [10][14][15].
4. **Valuation Concerns**: While some concerns exist about NVIDIA's valuation, the forward P/E ratio of 25 times forward earnings is considered reasonable given its growth trajectory [21][23].

### Conclusion

Our internal thesis on NVIDIA aligns well with the current market sentiment. Both emphasize NVIDIA's leadership in AI, robust financial growth, and strategic positioning. The market's bullish outlook, supported by strong analyst ratings and financial performance, reaffirms our "STRONG BUY" rating and price target. While valuation concerns are noted, they are outweighed by NVIDIA's growth potential and strategic advancements. Therefore, our thesis remains valid and well-supported by the latest market sentiment.

---

## 📊 Observability

View detailed traces in LangSmith:
- Each query shows the **tool call sequence**
- See which tools were checked first vs escalated
- Monitor **latency** differences between internal vs external calls
- Track **token usage** for cost optimization

**Dashboard:** https://smith.langchain.com


## 🔟 Key Benefits of This Architecture

1. **Internal-first** — Fast answers from DB and vector store; escalate only when external research is needed.
2. **Full citations** — Research Agent returns inline [1], [2] and numbered sources; display via `display_citations()`.
3. **Production-ready** — Research client has retry, stream timeout, and full chat_id logging.
4. **Reusability** — `create_financial_database()`, `create_vector_store()`, and tools come from `langgraph_core.py`.
5. **Observability** — LangSmith traces show when the agent uses internal vs Research Agent tools.

## 🎯 Next Steps

- **Add Search API** — Combine with `get_bigdata_tools()` (Search + KG) for news/filings before escalating to Research Agent (see `agent_to_search.ipynb`).
- **Tune escalation** — Adjust system prompt so the agent escalates only for “deep research” or “recent market” questions.
- **Research effort** — Use `research_effort="lite"` for faster, lighter answers or `"standard"` for full depth.
- **Follow-up** — Use `result.chat_id` and `client.follow_up()` for multi-turn Research Agent conversations.
- **LangSmith** — Use traces to see tool order and Research Agent usage.

---

## 📚 Additional Resources

- **Bigdata.com API docs**: https://docs.bigdata.com
- **Research Agent client**: `Research_Agent_Sync_Response/README.md` (retry, logging, chat_id)
- **LangGraph / LangChain**: https://langchain-ai.github.io/langgraph/
- **LangSmith**: https://smith.langchain.com
- **Agent_To_BigData README**: `README.md` in this folder

**Questions?** support@bigdata.com